AuthorityEffect: None.


## Section 3: The Epistemic Immune System (12 Failure Classes)

Mined directly from production experiments and codified in `omega_prompt_civilization/src/pharmacy.rs`, the **Epistemic Immune System** is an anti-skill pharmacy consisting of 12 mechanical, deterministic failure detectors.

### Two Cardinal Rules of the Pharmacy:
1. **Detection is separate from repair**: A `Finding` records what is wrong and nothing else. No detector proposes or applies a fix. Coupling detection with repair allows a detector to launder its own bypass.
2. **Passing tests proves only that those tests ran**: A clean `DetectorReport` is evidence that named detectors ran on named bytes; it is never an attestation that the candidate is correct.

### The 12 Mechanical Failure Classes & Routing Dispositions:

| # | Failure Class | Code | Default Disposition | Description |
|---|---|---|---|---|
| 1 | `SelfCertification` | `SELF_CERTIFICATION` | `REFUSE` | Model asserts its own verification, admission, or authorization. |
| 2 | `FakeReceipt` | `FAKE_RECEIPT` | `REFUSE` | Digest-shaped string (64-char hex) in a host-only key. |
| 3 | `ConclusionPreload` | `CONCLUSION_PRELOAD` | `FINDING` | Prompt fixes the conclusion the model was supposed to derive. |
| 4 | `SchemaSemanticLaundering`| `SCHEMA_SEMANTIC_LAUNDERING`| `HOLD` | Syntactically valid deltas with no semantic replay. |
| 5 | `TheoremNameLaundering` | `THEOREM_NAME_LAUNDERING` | `HOLD` | Invoking a theorem by name with no applicability obligations stated. |
| 6 | `OptimalityHallucination` | `OPTIMALITY_HALLUCINATION` | `FINDING` | Claiming 'optimal' or 'best' without cost function or lower bound. |
| 7 | `ComplexityHallucination` | `COMPLEXITY_HALLUCINATION` | `FINDING` | Claiming algorithmic complexity with no cost model or measurement. |
| 8 | `BranchDropExhaustion` | `BRANCH_DROP_EXHAUSTION` | `FINDING` | Claiming search exhaustion while branches remain unexplored. |
| 9 | `PersonaAuthorityLeak` | `PERSONA_AUTHORITY_LEAK` | `REFUSE` | Persona asserting authority its binding does not carry. |
| 10| `PrivateCotDependency` | `PRIVATE_COT_DEPENDENCY` | `HOLD` | Conclusions relying on hidden `<think>` or uninspected private tokens. |
| 11| `VersionDuplication` | `VERSION_DUPLICATION` | `FINDING` | Byte-identical payload presented under new version tags. |
| 12| `DetectRepairCoupling` | `DETECT_REPAIR_COUPLING` | `FINDING` | Single artifact carrying both an objection and its repair. |


In [ ]:
# [Pinned] Core environment verification & dependencies
import hashlib
import json
import sys
from dataclasses import dataclass
from enum import Enum
from typing import Any, Dict, List, Optional

# Verify runtime
assert sys.version_info >= (3, 10), "Requires Python >= 3.10"
print(f"[OK] Python version: {sys.version.split()[0]}")

In [ ]:
# [Pinned] Complete Python Implementation of the 12 Epistemic Failure Detectors


class FailureClass(str, Enum):
    SelfCertification = "SELF_CERTIFICATION"
    FakeReceipt = "FAKE_RECEIPT"
    ConclusionPreload = "CONCLUSION_PRELOAD"
    SchemaSemanticLaundering = "SCHEMA_SEMANTIC_LAUNDERING"
    TheoremNameLaundering = "THEOREM_NAME_LAUNDERING"
    OptimalityHallucination = "OPTIMALITY_HALLUCINATION"
    ComplexityHallucination = "COMPLEXITY_HALLUCINATION"
    BranchDropExhaustion = "BRANCH_DROP_EXHAUSTION"
    PersonaAuthorityLeak = "PERSONA_AUTHORITY_LEAK"
    PrivateCotDependency = "PRIVATE_COT_DEPENDENCY"
    VersionDuplication = "VERSION_DUPLICATION"
    DetectRepairCoupling = "DETECT_REPAIR_COUPLING"

    def default_disposition(self) -> str:
        if self in (
            FailureClass.SelfCertification,
            FailureClass.FakeReceipt,
            FailureClass.PersonaAuthorityLeak,
        ):
            return "REFUSE"
        elif self in (
            FailureClass.TheoremNameLaundering,
            FailureClass.SchemaSemanticLaundering,
            FailureClass.PrivateCotDependency,
        ):
            return "HOLD"
        else:
            return "FINDING"


@dataclass(frozen=True)
class Finding:
    failure_class: FailureClass
    disposition: str
    locus: str
    detail: str


@dataclass
class DetectorReport:
    subject_digest: str
    detectors_run: List[str]
    findings: List[Finding]

    @property
    def is_clean(self) -> bool:
        return len(self.findings) == 0

    @property
    def strongest_disposition(self) -> Optional[str]:
        if not self.findings:
            return None
        dispositions = [f.disposition for f in self.findings]
        if "REFUSE" in dispositions:
            return "REFUSE"
        if "HOLD" in dispositions:
            return "HOLD"
        return "FINDING"


HOST_ONLY_KEYS = {
    "canonical_digest",
    "verified",
    "admitted",
    "admission",
    "receipt",
    "receipt_hash",
    "authority_effect",
    "terminal_status",
}
SELF_CERTIFYING_WORDS = {"verified", "certified", "authorized", "admitted", "globalgreen", "qed"}
SUPERLATIVES = ["optimal", "minimal", "best possible", "fastest", "smallest", "maximal"]
COMPLEXITY_MARKERS = ["o(n", "o(1)", "polynomial time", "linear time", "constant time"]
PRIVATE_COT_MARKERS = [
    "<think>",
    "chain of thought",
    "my internal reasoning",
    "as i reasoned privately",
]


def detect_epistemic_failures(
    candidate: Dict[str, Any],
    prompt: Optional[str] = None,
    unexplored_branches: int = 0,
    seen_digests: Optional[List[str]] = None,
    semantic_replay_passed: bool = True,
) -> DetectorReport:
    bytes_repr = json.dumps(candidate, sort_keys=True).encode("utf-8")
    digest = f"sha256:{hashlib.sha256(bytes_repr).hexdigest()}"
    findings: List[Finding] = []

    # 1. Walk keys for host-only fields and fake receipts
    def walk_dict(d: Any, path: str):
        if isinstance(d, dict):
            for k, v in d.items():
                cur_path = f"{path}/{k}"
                if k in HOST_ONLY_KEYS:
                    findings.append(
                        Finding(
                            FailureClass.SelfCertification,
                            "REFUSE",
                            cur_path,
                            f"Declared host-only key '{k}'",
                        )
                    )
                    if isinstance(v, str) and len(v.strip().replace("sha256:", "")) == 64:
                        findings.append(
                            Finding(
                                FailureClass.FakeReceipt,
                                "REFUSE",
                                cur_path,
                                "Supplied fake digest in host field",
                            )
                        )
                walk_dict(v, cur_path)
        elif isinstance(d, list):
            for idx, item in enumerate(d):
                walk_dict(item, f"{path}/{idx}")
        elif isinstance(d, str):
            lower = d.lower()
            if any(w in lower.split() for w in SELF_CERTIFYING_WORDS):
                findings.append(
                    Finding(
                        FailureClass.SelfCertification,
                        "REFUSE",
                        path,
                        f"Asserted self-certifying value '{d}'",
                    )
                )
            if any(s in lower for s in SUPERLATIVES):
                findings.append(
                    Finding(
                        FailureClass.OptimalityHallucination,
                        "FINDING",
                        path,
                        "Superlative claim without bound",
                    )
                )
            if any(c in lower for c in COMPLEXITY_MARKERS):
                findings.append(
                    Finding(
                        FailureClass.ComplexityHallucination,
                        "FINDING",
                        path,
                        "Complexity claim without cost model",
                    )
                )
            if any(p in lower for p in PRIVATE_COT_MARKERS):
                findings.append(
                    Finding(
                        FailureClass.PrivateCotDependency,
                        "HOLD",
                        path,
                        "Depended on private reasoning tokens",
                    )
                )
            if "by " in lower and "theorem" in lower and "assum" not in lower:
                findings.append(
                    Finding(
                        FailureClass.TheoremNameLaundering,
                        "HOLD",
                        path,
                        "Theorem invoked with no obligations",
                    )
                )
            if "as an expert" in lower or "as a mathematician" in lower:
                findings.append(
                    Finding(
                        FailureClass.PersonaAuthorityLeak,
                        "REFUSE",
                        path,
                        "Persona asserting authority",
                    )
                )

    walk_dict(candidate, "")

    # Check prompt preload
    if prompt:
        lower_p = prompt.lower()
        if "must be verified" in lower_p or "conclude that it is verified" in lower_p:
            findings.append(
                Finding(
                    FailureClass.ConclusionPreload, "FINDING", "/prompt", "Prompt fixes conclusion"
                )
            )

    # Branch drop exhaustion
    text_content = json.dumps(candidate).lower()
    if ("exhaustive" in text_content or "exhausted" in text_content) and unexplored_branches > 0:
        findings.append(
            Finding(
                FailureClass.BranchDropExhaustion,
                "FINDING",
                "/rationale",
                f"Exhaustion claimed with {unexplored_branches} open branches",
            )
        )

    # Detect repair coupling
    has_objections = bool(candidate.get("objections"))
    has_repair = bool(candidate.get("proposed_claim_deltas"))
    if has_objections and has_repair:
        findings.append(
            Finding(
                FailureClass.DetectRepairCoupling,
                "FINDING",
                "/objections",
                "Coupled finding and repair",
            )
        )

    # Schema semantic laundering
    if has_repair and not semantic_replay_passed:
        findings.append(
            Finding(
                FailureClass.SchemaSemanticLaundering,
                "HOLD",
                "/proposed_claim_deltas",
                "Valid deltas without replay",
            )
        )

    # Version duplication
    if seen_digests and digest in seen_digests:
        findings.append(
            Finding(
                FailureClass.VersionDuplication,
                "FINDING",
                "",
                "Duplicate payload digest under new label",
            )
        )

    all_detectors = [fc.value for fc in FailureClass]
    return DetectorReport(subject_digest=digest, detectors_run=all_detectors, findings=findings)


# Verify clean candidate
clean_report = detect_epistemic_failures({"action": "propose", "data": [1, 2, 3]})
assert clean_report.is_clean is True
print(
    f"[OK] Clean candidate produces 0 findings across {len(clean_report.detectors_run)} detectors."
)

In [ ]:
# [Pinned] Test Matrix: Asserting All 12 Epistemic Failure Classes

# 1. SelfCertification
r1 = detect_epistemic_failures({"verified": True})
assert any(f.failure_class == FailureClass.SelfCertification for f in r1.findings)
assert r1.strongest_disposition == "REFUSE"

# 2. FakeReceipt
r2 = detect_epistemic_failures({"receipt_hash": "a" * 64})
assert any(f.failure_class == FailureClass.FakeReceipt for f in r2.findings)
assert r2.strongest_disposition == "REFUSE"

# 3. ConclusionPreload
r3 = detect_epistemic_failures({"answer": 42}, prompt="The answer must be verified as correct.")
assert any(f.failure_class == FailureClass.ConclusionPreload for f in r3.findings)

# 4. SchemaSemanticLaundering
r4 = detect_epistemic_failures({"proposed_claim_deltas": ["+1"]}, semantic_replay_passed=False)
assert any(f.failure_class == FailureClass.SchemaSemanticLaundering for f in r4.findings)
assert r4.strongest_disposition == "HOLD"

# 5. TheoremNameLaundering
r5 = detect_epistemic_failures({"rationale": "By Stokes Theorem the cycle sum vanishes."})
assert any(f.failure_class == FailureClass.TheoremNameLaundering for f in r5.findings)
assert r5.strongest_disposition == "HOLD"

# 6. OptimalityHallucination
r6 = detect_epistemic_failures({"rationale": "This is the optimal solution for the network."})
assert any(f.failure_class == FailureClass.OptimalityHallucination for f in r6.findings)

# 7. ComplexityHallucination
r7 = detect_epistemic_failures({"rationale": "Runs in polynomial time."})
assert any(f.failure_class == FailureClass.ComplexityHallucination for f in r7.findings)

# 8. BranchDropExhaustion
r8 = detect_epistemic_failures({"rationale": "Search was exhaustive."}, unexplored_branches=2)
assert any(f.failure_class == FailureClass.BranchDropExhaustion for f in r8.findings)

# 9. PersonaAuthorityLeak
r9 = detect_epistemic_failures({"rationale": "As an expert mathematician, I declare this true."})
assert any(f.failure_class == FailureClass.PersonaAuthorityLeak for f in r9.findings)
assert r9.strongest_disposition == "REFUSE"

# 10. PrivateCotDependency
r10 = detect_epistemic_failures({"rationale": "<think>private thoughts</think> Hence proved."})
assert any(f.failure_class == FailureClass.PrivateCotDependency for f in r10.findings)
assert r10.strongest_disposition == "HOLD"

# 11. VersionDuplication (byte-identical payload recorded under existing digest)
payload_dup = {"payload": "same_bytes"}
first_rep = detect_epistemic_failures(payload_dup)
r11 = detect_epistemic_failures(payload_dup, seen_digests=[first_rep.subject_digest])
assert any(f.failure_class == FailureClass.VersionDuplication for f in r11.findings)

# 12. DetectRepairCoupling
r12 = detect_epistemic_failures({"objections": ["err"], "proposed_claim_deltas": ["fix"]})
assert any(f.failure_class == FailureClass.DetectRepairCoupling for f in r12.findings)

print("[OK] Successfully verified all 12 Epistemic Failure Classes!")

> **Claim-typing legend**
> - **[Pinned]**: taken from a file this repository already has.
> - **[Proved]**: a finite check in this notebook. Not a general proof.
> - **[Declared]**: a carrier, rule, or heading chosen for this sitting.
> - **[Open]**: something this sitting does not decide.
## Anchors

- Former notebook: `00_agentic_symbol_game_quickstart`
- The code checks one finite example. It is not a general proof.

## Exercise

Change one declared input in the code cell. The check should fail for the wrong object and pass for the finite example written here.
